In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/


In [ ]:
!kaggle datasets download -d mlg-ulb/creditcardfraud

Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
  0% 0.00/66.0M [00:00<?, ?B/s]
100% 66.0M/66.0M [00:00<00:00, 1.67GB/s]


In [ ]:
import zipfile
zip_ref=zipfile.ZipFile('/content/creditcardfraud.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import pandas as pd
df=pd.read_csv('/content/creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
df['Class'].value_counts()

,count
Class,
0,284315
1,492


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x=df.drop('Class',axis=1)
y=df['Class']

scaler= StandardScaler()
x=scaler.fit_transform(x)

x_train,x_test,y_train,y_test =train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
x_train.shape,x_test.shape

((227845, 30), (56962, 30))

In [ ]:
y_test.shape,y_train.shape

((56962,), (227845,))

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
x_train_reshamped,y_train_reshamped= smote.fit_resample(x_train,y_train)

In [ ]:
# from sklearn.ensemble import RandomForestClassifier

# rfc = RandomForestClassifier(random_state=42,n_estimators=100)
# rfc.fit(x_train_reshamped,y_train_reshamped)

In [ ]:
# from xgboost import XGBClassifier
# import xgboost

# xg=XGBClassifier(random_state=42,n_estimators=100)
# xg.fit(x_train_reshamped,y_train_reshamped)

In [ ]:
from sklearn.ensemble import RandomForestClassifier,VotingClassifier

rfc = RandomForestClassifier(random_state=42,n_estimators=100)
rfc.fit(x_train_reshamped,y_train_reshamped)

from xgboost import XGBClassifier
import xgboost

xg=XGBClassifier(random_state=42,n_estimators=100)
xg.fit(x_train_reshamped,y_train_reshamped)

ensemble = VotingClassifier(estimators=[('rfc',rfc),('xg',xg)],
                            voting='soft')
ensemble.fit(x_train_reshamped,y_train_reshamped)


VotingClassifier(estimators=[('rfc', RandomForestClassifier(random_state=42)),
                             ('xg',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=100, n_jobs=None,
                                            num_parallel_tree=None, ...))],
                 voting='soft')

In [ ]:
# y_pred=rfc.predict(x_test)

y_pred=xg.predict(x_test)



In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.metrics import roc_auc_score

print(roc_auc_score(y_test, y_pred))
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))

0.9334976111997978
[[56844    20]
 [   13    85]]
0.999420666409185


In [ ]:
import joblib

joblib.dump(ensemble,'credit_card_model.pkl')
joblib.dump(scaler,'scaler.pkl')

['scaler.pkl']